# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [ ]:
!pwd

/content/Ultramarin


In [1]:
!ls /content

drive  sample_data  Ultramarin


In [2]:
import os, sys, subprocess

# Get project files onto remote runtime (Colab extension doesn't sync files)
if not os.path.isdir("/content/Ultramarin/utils"):
    subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                    "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                   cwd="/content")
os.chdir("/content/Ultramarin")
sys.path.insert(0, os.getcwd())

import numpy as np
from pathlib import Path
import pandas as pd
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val
from utils.test import generate_test_loader, generate_test_predictions
from data.simulate_walk_the_book import simulate_walk_the_book
import warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

device: cuda


In [3]:
# Copy data from Google Drive (data/ is gitignored, parquets don't sync)
import os, subprocess

# Always target /content/Ultramarin/data regardless of current working directory
dst = "/content/Ultramarin/data"

if not os.path.isfile(os.path.join(dst, "BTCUSDT/X_train.parquet")):
    print("Data not found. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

    src = "/content/drive/MyDrive/data"
    os.makedirs(dst, exist_ok=True)
    print(f"Copying from {src} to {dst} ...")

    result = subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR copying data: {result.stderr}")
    else:
        print("Data copied successfully.")
    
    # Verify
    print(f"data/ contents: {os.listdir(dst)}")
    btc_dir = os.path.join(dst, "BTCUSDT")
    if os.path.isdir(btc_dir):
        print(f"data/BTCUSDT/ contents: {os.listdir(btc_dir)}")
else:
    print("Data already present.")

Data already present.


In [4]:
# Paths and volume_to_fill
from pathlib import Path

root = Path("data") if Path("data").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

data_asset = "BTCUSDT"
symbol_dir = root / data_asset
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
X_test_path = symbol_dir / "X_test.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

# Optimal K per currency (from all-currency K-sweep on full training data)
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}
K_SECONDS = OPTIMAL_K.get(data_asset, 14)

# Known volumes per currency (fallback if vol_to_fill.txt is missing)
KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))

if volume_to_fill is None:
    volume_to_fill = KNOWN_VOLUMES.get(data_asset)
    print(f"vol_to_fill.txt not found, using known volume: {volume_to_fill}")
else:
    print(f"volume_to_fill: {volume_to_fill}")

print(f"K_SECONDS: {K_SECONDS} (optimal for {data_asset})")

volume_to_fill: 4.0
K_SECONDS: 14 (optimal for BTCUSDT)


In [5]:
# DeepLOB only take LOB features as input
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

OFI_COLS = ['ofi_1', 'ofi_2', 'ofi_3', 'ofi_4', 'ofi_5', 'ofi_agg']

FEATURE_COLS = LOB_COLS + ["mid_price"] + OFI_COLS

# Verification print
print(f"LOB Features: {len(LOB_COLS)}")
print(f"Extra Features: mid_price + {len(OFI_COLS)} OFI = {1 + len(OFI_COLS)}")
print(f"Total Features: {len(FEATURE_COLS)} (20 LOB + 1 mid_price + 6 OFI)")

LOB Features: 20
Extra Features: mid_price + 6 OFI = 7
Total Features: 27 (20 LOB + 1 mid_price + 6 OFI)


In [6]:
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# Model Training

In [7]:
# --- Execution ---
config = TrainCfg(

    # Hyperparameters
    epochs = 100,
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.01,
    direction_lambda = 0.1,
    dropout = 0.1,

    # Scheduled teacher forcing
    tf_floor = 0.1,

    # Early stopping
    patience = 10,
    min_lr = 1e-6,
    
    # Windows
    input_window = 600,   # Look-back
    target_window = 60,   # Horizon
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    x_test_path = X_test_path,
    feature_cols = FEATURE_COLS,
)

model, train_scalers, val_loader, val_id_map, processor = train_val(config)

Epoch 01 | Train: 0.892946 | Val: 0.968003 | LR: 1.00e-03 | TF: 1.00
Epoch 02 | Train: 0.269588 | Val: 1.692971 | LR: 9.99e-04 | TF: 0.99
Epoch 03 | Train: 0.102318 | Val: 1.395863 | LR: 9.98e-04 | TF: 0.98
Epoch 04 | Train: 0.053784 | Val: 1.581101 | LR: 9.96e-04 | TF: 0.97
Epoch 05 | Train: 0.038948 | Val: 0.794814 | LR: 9.94e-04 | TF: 0.96
Epoch 06 | Train: 0.031069 | Val: 0.307914 | LR: 9.91e-04 | TF: 0.95
Epoch 07 | Train: 0.025071 | Val: 0.281653 | LR: 9.88e-04 | TF: 0.94
Epoch 08 | Train: 0.022826 | Val: 0.371620 | LR: 9.84e-04 | TF: 0.93
Epoch 09 | Train: 0.021055 | Val: 0.318741 | LR: 9.80e-04 | TF: 0.92
Epoch 10 | Train: 0.019767 | Val: 0.754429 | LR: 9.76e-04 | TF: 0.91
Epoch 11 | Train: 0.018711 | Val: 0.284183 | LR: 9.70e-04 | TF: 0.90
Epoch 12 | Train: 0.016970 | Val: 0.324427 | LR: 9.65e-04 | TF: 0.89
Epoch 13 | Train: 0.016177 | Val: 0.560729 | LR: 9.59e-04 | TF: 0.88
Epoch 14 | Train: 0.016037 | Val: 0.175455 | LR: 9.52e-04 | TF: 0.87
Epoch 15 | Train: 0.015441 | Val: 

# Model-Based Predictions

## Cell

In [8]:
# Implementation error for the model

import pandas as pd
import numpy as np
from data.simulate_walk_the_book import simulate_walk_the_book

# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in val_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions: directional + windowed
    # Buy more when predicted price is below predicted close (cheaper)
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    window_preds = mid_preds[-K_SECONDS:]
    favorability = pred_close - window_preds  # positive = cheaper = buy more
    temp = np.std(favorability) + 1e-8
    weights = np.exp(favorability / temp)
    weights /= weights.sum()
    
    model_positions = np.zeros(60)
    model_positions[-K_SECONDS:] = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_bps.append(impl_error * vol_penalty)

model_bps = np.array(model_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR (directional + windowed)")
print(f"{'='*50}")
print(f"Window: last {K_SECONDS}s, directional softmax scoring")
print(f"Instruments evaluated: {len(model_bps)}")
print(f"Mean:   {model_bps.mean():.4f} bps")
print(f"Median: {np.median(model_bps):.4f} bps")
print(f"Std:    {model_bps.std():.4f} bps")
print(f"Min:    {model_bps.min():.4f} bps")
print(f"Max:    {model_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (141, 60)
Number of validation instruments: 141

MODEL IMPLEMENTATION ERROR (directional + windowed)
Window: last 14s, directional softmax scoring
Instruments evaluated: 141
Mean:   1.5387 bps
Median: 1.2441 bps
Std:    1.2804 bps
Min:    0.0000 bps
Max:    10.2943 bps


## Positions

In [9]:
times = pd.read_parquet(Y_path)["time_in_hour"].sort_values(ascending=True).unique()

In [10]:
# --- 1. Generate predictions using test_loader ---
model.eval()
test_preds = []

test_loader, scalers, test_map, processor = generate_test_loader(config, processor)

with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None)
        test_preds.append(pred.cpu().numpy())

test_preds = np.concatenate(test_preds, axis=0)  # Shape: [num_test_ids, 60]

# Get test IDs
test_ids = [int(uid) for idx, uid in test_map.cpu().numpy()]
print(f"Test predictions shape: {test_preds.shape}")
print(f"Number of testing instruments: {len(test_ids)}")

test_positions = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Model-based positions: directional + windowed
    mid_preds = test_preds[i]
    pred_close = mid_preds[-1]
    
    window_preds = mid_preds[-K_SECONDS:]
    favorability = pred_close - window_preds  # positive = cheaper = buy more
    temp = np.std(favorability) + 1e-8
    weights = np.exp(favorability / temp)
    weights /= weights.sum()
    
    model_positions = np.zeros(60)
    model_positions[-K_SECONDS:] = weights * volume_to_fill

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': model_positions
    })

    test_positions = pd.concat([test_positions, df], ignore_index=True)

test_positions

Test predictions shape: (303, 60)
Number of testing instruments: 303


,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.000000
1,5.692284e+16,0 days 00:59:01,0.000000
2,5.692284e+16,0 days 00:59:02,0.000000
3,5.692284e+16,0 days 00:59:03,0.000000
4,5.692284e+16,0 days 00:59:04,0.000000
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.338913
18176,1.841373e+19,0 days 00:59:56,0.432937
18177,1.841373e+19,0 days 00:59:57,0.552458
18178,1.841373e+19,0 days 00:59:58,0.704238


In [11]:
weights.sum()

np.float32(1.0000001)

# TWAP

## Cell

In [12]:
# Implementation error for the baseline (TWAP)

baseline_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Baseline TWAP positions
    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS
    
    # Simulate
    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_bps.append(impl_error * vol_penalty)

baseline_bps = np.array(baseline_bps)

print(f"\n{'='*50}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(baseline_bps)}")
print(f"Mean:   {baseline_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_bps):.4f} bps")
print(f"Std:    {baseline_bps.std():.4f} bps")
print(f"Min:    {baseline_bps.min():.4f} bps")
print(f"Max:    {baseline_bps.max():.4f} bps")
print(f"{'='*50}")


BASELINE (TWAP-14s) IMPLEMENTATION ERROR
Instruments evaluated: 141
Mean:   1.3232 bps
Median: 1.1836 bps
Std:    1.0940 bps
Min:    0.0000 bps
Max:    9.0855 bps


In [13]:
# === TWAP K-Sweep: Find optimal trading window ===

sweep_results = {}

for K in range(1, 61):
    k_bps = []
    for i, anon_id in enumerate(val_ids):
        df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
        if len(df_inst) != 60:
            continue

        ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
        ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
        bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
        bid_vols = df_inst[BID_VOL_COLS].to_numpy()
        close_price = df_inst['close'].dropna().iloc[-1]

        positions = np.zeros(60)
        positions[-K:] = volume_to_fill / K

        vol, avg_price = simulate_walk_the_book(
            positions, ask_prices, ask_vols, bid_prices, bid_vols
        )

        if vol > 0 and not np.isnan(avg_price):
            ie = np.abs(avg_price - close_price) / close_price * 10000
            vp = min(100.0, volume_to_fill / vol)
            k_bps.append(ie * vp)

    sweep_results[K] = np.mean(k_bps)

# Find optimal K
best_K = min(sweep_results, key=sweep_results.get)
best_bps = sweep_results[best_K]

print(f"{'='*50}")
print(f"TWAP K-SWEEP for {data_asset} (vol={volume_to_fill})")
print(f"{'='*50}")
print(f"Best K = {best_K} → {best_bps:.4f} bps")
print(f"Current K = {K_SECONDS} → {sweep_results[K_SECONDS]:.4f} bps")
print(f"K=60 (full minute) → {sweep_results[60]:.4f} bps")
print(f"K=1  (last second) → {sweep_results[1]:.4f} bps")
print()
print("Top 10 K values:")
for k, bps in sorted(sweep_results.items(), key=lambda x: x[1])[:10]:
    marker = " <<<" if k == best_K else ""
    print(f"  K={k:2d} → {bps:.4f} bps{marker}")

TWAP K-SWEEP for BTCUSDT (vol=4.0)
Best K = 11 → 1.3123 bps
Current K = 14 → 1.3232 bps
K=60 (full minute) → 1.7176 bps
K=1  (last second) → 9.4701 bps

Top 10 K values:
  K=11 → 1.3123 bps <<<
  K=10 → 1.3143 bps
  K=12 → 1.3176 bps
  K=13 → 1.3211 bps
  K=14 → 1.3232 bps
  K= 9 → 1.3329 bps
  K=15 → 1.3376 bps
  K=16 → 1.3545 bps
  K=17 → 1.3624 bps
  K=18 → 1.3670 bps


In [14]:
# === ALL-CURRENCY K-SWEEP ===
# Doesn't need the model — just raw Y data + walk-the-book simulator

from pathlib import Path
from data.simulate_walk_the_book import simulate_walk_the_book

ALL_CURRENCIES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}

ASK_P = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_V = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_P = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_V = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

optimal_k = {}

for currency, vol in ALL_CURRENCIES.items():
    y_path = Path(f"data/{currency}/y_train.parquet")
    if not y_path.exists():
        print(f"SKIP {currency}: {y_path} not found")
        continue

    Y = pd.read_parquet(y_path).sort_values(["anonymized_id", "time_in_hour"])
    all_ids = Y["anonymized_id"].unique()

    # Pre-extract per-instrument data to avoid repeated filtering
    inst_data = []
    for anon_id in all_ids:
        df_inst = Y[Y["anonymized_id"] == anon_id].sort_values("time_in_hour")
        if len(df_inst) != 60:
            continue
        inst_data.append({
            "ask_prices": df_inst[ASK_P].to_numpy(),
            "ask_vols": df_inst[ASK_V].to_numpy(),
            "bid_prices": df_inst[BID_P].to_numpy(),
            "bid_vols": df_inst[BID_V].to_numpy(),
            "close": df_inst['close'].dropna().iloc[-1],
        })

    sweep = {}
    for K in range(1, 61):
        k_bps = []
        for inst in inst_data:
            positions = np.zeros(60)
            positions[-K:] = vol / K

            filled_vol, avg_price = simulate_walk_the_book(
                positions, inst["ask_prices"], inst["ask_vols"],
                inst["bid_prices"], inst["bid_vols"]
            )

            if filled_vol > 0 and not np.isnan(avg_price):
                ie = np.abs(avg_price - inst["close"]) / inst["close"] * 10000
                vp = min(100.0, vol / filled_vol)
                k_bps.append(ie * vp)

        sweep[K] = np.mean(k_bps)

    best_k = min(sweep, key=sweep.get)
    optimal_k[currency] = (best_k, sweep[best_k], sweep[14])

    print(f"\n{currency} (vol={vol}, {len(inst_data)} instruments)")
    print(f"  Best K={best_k} → {sweep[best_k]:.4f} bps")
    print(f"  K=14   → {sweep[14]:.4f} bps  (delta: {sweep[14] - sweep[best_k]:+.4f})")
    top5 = sorted(sweep.items(), key=lambda x: x[1])[:5]
    print(f"  Top 5: {', '.join(f'K={k}({b:.4f})' for k, b in top5)}")

print(f"\n{'='*60}")
print(f"OPTIMAL K PER CURRENCY")
print(f"{'='*60}")
print(f"{'Currency':<12} {'Volume':>10} {'Best K':>8} {'Best BPS':>10} {'K=14 BPS':>10} {'Saved':>8}")
print(f"{'-'*60}")
for currency, (k, best_bps, k14_bps) in sorted(optimal_k.items(), key=lambda x: x[1][0]):
    saved = k14_bps - best_bps
    print(f"{currency:<12} {ALL_CURRENCIES[currency]:>10.0f} {k:>8d} {best_bps:>10.4f} {k14_bps:>10.4f} {saved:>+8.4f}")


BTCUSDT (vol=4.0, 705 instruments)
  Best K=14 → 1.2186 bps
  K=14   → 1.2186 bps  (delta: +0.0000)
  Top 5: K=14(1.2186), K=15(1.2188), K=13(1.2208), K=16(1.2225), K=17(1.2246)

ETHUSDT (vol=55.0, 703 instruments)
  Best K=26 → 2.6624 bps
  K=14   → 2.7876 bps  (delta: +0.1252)
  Top 5: K=26(2.6624), K=25(2.6627), K=22(2.6651), K=27(2.6651), K=28(2.6652)

LTCUSDT (vol=51.0, 389 instruments)
  Best K=16 → 4.8966 bps
  K=14   → 4.9388 bps  (delta: +0.0422)
  Top 5: K=16(4.8966), K=17(4.9053), K=18(4.9097), K=10(4.9196), K=15(4.9229)

SOLUSDT (vol=315.0, 693 instruments)
  Best K=17 → 5.2657 bps
  K=14   → 5.2940 bps  (delta: +0.0284)
  Top 5: K=17(5.2657), K=18(5.2669), K=20(5.2758), K=19(5.2763), K=16(5.2779)

ADAUSDT (vol=10436.0, 418 instruments)
  Best K=7 → 4.8692 bps
  K=14   → 4.9459 bps  (delta: +0.0767)
  Top 5: K=7(4.8692), K=10(4.8786), K=12(4.8836), K=11(4.8966), K=6(4.8990)

DOGEUSDT (vol=60349.0, 594 instruments)
  Best K=20 → 4.5620 bps
  K=14   → 4.6373 bps  (delta: +0.

In [15]:
# === DIAGNOSTIC: Oracle ceiling + model signal quality ===

from scipy.stats import spearmanr

oracle_bps = []
oracle_full_bps = []  # Oracle using ALL 60 seconds
rank_correlations = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    actual_mid = ((df_inst['ask_price_1'] + df_inst['bid_price_1']) / 2).to_numpy()
    
    # --- Oracle (windowed): last K seconds only ---
    actual_window = actual_mid[-K_SECONDS:]
    actual_close = actual_mid[-1]
    oracle_fav = actual_close - actual_window
    oracle_temp = np.std(oracle_fav) + 1e-8
    oracle_w = np.exp(oracle_fav / oracle_temp)
    oracle_w /= oracle_w.sum()
    oracle_pos = np.zeros(60)
    oracle_pos[-K_SECONDS:] = oracle_w * volume_to_fill
    
    vol, avg_price = simulate_walk_the_book(oracle_pos, ask_prices, ask_vols, bid_prices, bid_vols)
    if vol > 0 and not np.isnan(avg_price):
        ie = np.abs(avg_price - close_price) / close_price * 10000
        vp = min(100.0, volume_to_fill / vol)
        oracle_bps.append(ie * vp)
    
    # --- Oracle (full 60s): use ALL seconds with perfect foresight ---
    oracle_fav_full = actual_close - actual_mid  # all 60 seconds
    oracle_temp_full = np.std(oracle_fav_full) + 1e-8
    oracle_w_full = np.exp(oracle_fav_full / oracle_temp_full)
    oracle_w_full /= oracle_w_full.sum()
    oracle_pos_full = oracle_w_full * volume_to_fill
    
    vol_f, avg_price_f = simulate_walk_the_book(oracle_pos_full, ask_prices, ask_vols, bid_prices, bid_vols)
    if vol_f > 0 and not np.isnan(avg_price_f):
        ie_f = np.abs(avg_price_f - close_price) / close_price * 10000
        vp_f = min(100.0, volume_to_fill / vol_f)
        oracle_full_bps.append(ie_f * vp_f)
    
    # --- Rank correlation: does model predict the right ordering? ---
    pred_window = val_preds[i][-K_SECONDS:]
    if np.std(pred_window) > 1e-10 and np.std(actual_window) > 1e-10:
        corr, _ = spearmanr(pred_window, actual_window)
        rank_correlations.append(corr)

oracle_bps = np.array(oracle_bps)
oracle_full_bps = np.array(oracle_full_bps)
rank_correlations = np.array(rank_correlations)

print(f"{'='*60}")
print(f"DIAGNOSTIC: How much headroom is there?")
print(f"{'='*60}")
print(f"")
print(f"Oracle FULL (60s, perfect foresight):     {oracle_full_bps.mean():.4f} bps")
print(f"Oracle windowed ({K_SECONDS}s, perfect foresight): {oracle_bps.mean():.4f} bps")
print(f"Model (directional + windowed):           {model_bps.mean():.4f} bps")
print(f"TWAP-{K_SECONDS}s:                                {baseline_bps.mean():.4f} bps")
print(f"")
print(f"Oracle FULL beats TWAP by:  {baseline_bps.mean() - oracle_full_bps.mean():.4f} bps")
print(f"Oracle {K_SECONDS}s beats TWAP by:   {baseline_bps.mean() - oracle_bps.mean():.4f} bps")
print(f"Model vs TWAP gap:          {model_bps.mean() - baseline_bps.mean():+.4f} bps")
print(f"")
print(f"Rank correlation (predicted vs actual price ordering in window):")
print(f"  Mean:   {rank_correlations.mean():.4f}")
print(f"  Median: {np.median(rank_correlations):.4f}")
print(f"  Std:    {rank_correlations.std():.4f}")
print(f"  >0:     {(rank_correlations > 0).mean()*100:.1f}% of instruments")
print(f"  >0.3:   {(rank_correlations > 0.3).mean()*100:.1f}% of instruments")
print(f"{'='*60}")

DIAGNOSTIC: How much headroom is there?

Oracle FULL (60s, perfect foresight):     1.6270 bps
Oracle windowed (14s, perfect foresight): 1.3076 bps
Model (directional + windowed):           1.5387 bps
TWAP-14s:                                1.3232 bps

Oracle FULL beats TWAP by:  -0.3038 bps
Oracle 14s beats TWAP by:   0.0156 bps
Model vs TWAP gap:          +0.2155 bps

Rank correlation (predicted vs actual price ordering in window):
  Mean:   0.0677
  Median: 0.1341
  Std:    0.5481
  >0:     56.7% of instruments
  >0.3:   39.0% of instruments


In [16]:
# === TRUE ORACLE: Optimal allocation with full order book knowledge ===
# Unlike the mid-price oracle (which only sees mid prices and uses a softmax formula),
# this oracle sees the FULL order book (all 5 levels of ask prices + volumes) at every
# second and uses scipy optimization to find the volume allocation that minimizes
# |VWAP - close|. This accounts for spread, market impact, and book depth.

from scipy.optimize import minimize

def walk_the_book_cost(volume, ask_prices_row, ask_vols_row):
    """Compute total cost of buying `volume` units at one second by walking the book.
    Returns (total_cost, volume_actually_filled)."""
    remaining = volume
    total_cost = 0.0
    filled = 0.0
    for level in range(len(ask_prices_row)):
        price = ask_prices_row[level]
        avail = ask_vols_row[level]
        if np.isnan(price) or np.isnan(avail):
            continue
        take = min(remaining, avail)
        total_cost += take * price
        filled += take
        remaining -= take
        if remaining <= 1e-12:
            break
    return total_cost, filled

def true_oracle_objective(v_alloc, ask_prices, ask_vols, close_price, vol_to_fill):
    """Objective: (VWAP - close)^2. v_alloc is the volume allocation per second."""
    total_cost = 0.0
    total_filled = 0.0
    for t in range(len(v_alloc)):
        if v_alloc[t] < 1e-12:
            continue
        cost, filled = walk_the_book_cost(v_alloc[t], ask_prices[t], ask_vols[t])
        total_cost += cost
        total_filled += filled
    if total_filled < 1e-12:
        return 1e12
    vwap = total_cost / total_filled
    # Also penalize unfilled volume
    fill_penalty = (vol_to_fill - total_filled) ** 2 * 1e6
    return (vwap - close_price) ** 2 + fill_penalty

true_oracle_bps = []
true_oracle_bps_full = []
midprice_oracle_bps_for_comparison = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    if len(df_inst) != 60:
        continue

    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]

    # --- True Oracle (windowed, last K seconds) ---
    # Initialize with TWAP allocation in window
    v0 = np.zeros(K_SECONDS)
    v0[:] = volume_to_fill / K_SECONDS

    ask_p_window = ask_prices[-K_SECONDS:]
    ask_v_window = ask_vols[-K_SECONDS:]

    result = minimize(
        true_oracle_objective,
        v0,
        args=(ask_p_window, ask_v_window, close_price, volume_to_fill),
        method='SLSQP',
        bounds=[(0, volume_to_fill)] * K_SECONDS,
        constraints={'type': 'eq', 'fun': lambda v: v.sum() - volume_to_fill},
        options={'maxiter': 500, 'ftol': 1e-15}
    )

    opt_alloc = np.zeros(60)
    opt_alloc[-K_SECONDS:] = result.x
    vol, avg_price = simulate_walk_the_book(opt_alloc, ask_prices, ask_vols, bid_prices, bid_vols)
    if vol > 0 and not np.isnan(avg_price):
        ie = np.abs(avg_price - close_price) / close_price * 10000
        vp = min(100.0, volume_to_fill / vol)
        true_oracle_bps.append(ie * vp)

    # --- True Oracle (full 60s) ---
    v0_full = np.zeros(60)
    v0_full[:] = volume_to_fill / 60

    result_full = minimize(
        true_oracle_objective,
        v0_full,
        args=(ask_prices, ask_vols, close_price, volume_to_fill),
        method='SLSQP',
        bounds=[(0, volume_to_fill)] * 60,
        constraints={'type': 'eq', 'fun': lambda v: v.sum() - volume_to_fill},
        options={'maxiter': 500, 'ftol': 1e-15}
    )

    opt_alloc_full = result_full.x
    vol_f, avg_price_f = simulate_walk_the_book(opt_alloc_full, ask_prices, ask_vols, bid_prices, bid_vols)
    if vol_f > 0 and not np.isnan(avg_price_f):
        ie_f = np.abs(avg_price_f - close_price) / close_price * 10000
        vp_f = min(100.0, volume_to_fill / vol_f)
        true_oracle_bps_full.append(ie_f * vp_f)

true_oracle_bps = np.array(true_oracle_bps)
true_oracle_bps_full = np.array(true_oracle_bps_full)

print(f"{'='*65}")
print(f"TRUE ORACLE vs MID-PRICE ORACLE vs TWAP")
print(f"{'='*65}")
print(f"")
print(f"TRUE Oracle (full 60s, optimal allocation):  {true_oracle_bps_full.mean():.4f} bps")
print(f"TRUE Oracle (windowed {K_SECONDS}s, optimal alloc):  {true_oracle_bps.mean():.4f} bps")
print(f"Mid-price Oracle (windowed {K_SECONDS}s, softmax):   {oracle_bps.mean():.4f} bps")
print(f"Mid-price Oracle (full 60s, softmax):        {oracle_full_bps.mean():.4f} bps")
print(f"TWAP-{K_SECONDS}s:                                   {baseline_bps.mean():.4f} bps")
print(f"Model (directional + windowed):              {model_bps.mean():.4f} bps")
print(f"")
print(f"TRUE Oracle 60s beats TWAP by:   {baseline_bps.mean() - true_oracle_bps_full.mean():.4f} bps")
print(f"TRUE Oracle {K_SECONDS}s beats TWAP by:  {baseline_bps.mean() - true_oracle_bps.mean():.4f} bps")
print(f"Mid-price Oracle {K_SECONDS}s beats TWAP: {baseline_bps.mean() - oracle_bps.mean():.4f} bps")
print(f"{'='*65}")

TRUE ORACLE vs MID-PRICE ORACLE vs TWAP

TRUE Oracle (full 60s, optimal allocation):  0.1591 bps
TRUE Oracle (windowed 14s, optimal alloc):  0.7267 bps
Mid-price Oracle (windowed 14s, softmax):   1.3076 bps
Mid-price Oracle (full 60s, softmax):        1.6270 bps
TWAP-14s:                                   1.3232 bps
Model (directional + windowed):              1.5387 bps

TRUE Oracle 60s beats TWAP by:   1.1641 bps
TRUE Oracle 14s beats TWAP by:  0.5965 bps
Mid-price Oracle 14s beats TWAP: 0.0156 bps


## Positions

In [17]:
test_positions_twap = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    # Match the TWAP-14s strategy used in evaluation
    twap_positions = np.zeros(60)
    twap_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': twap_positions
    })

    test_positions_twap = pd.concat([test_positions_twap, df], ignore_index=True)

test_positions_twap

,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.000000
1,5.692284e+16,0 days 00:59:01,0.000000
2,5.692284e+16,0 days 00:59:02,0.000000
3,5.692284e+16,0 days 00:59:03,0.000000
4,5.692284e+16,0 days 00:59:04,0.000000
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.285714
18176,1.841373e+19,0 days 00:59:56,0.285714
18177,1.841373e+19,0 days 00:59:57,0.285714
18178,1.841373e+19,0 days 00:59:58,0.285714


# Predictive_Scheduler Predictions

## Cell

In [18]:
# Implementation error for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Configure the scheduler — uses optimal K, directional scoring, conservative blend
sched_cfg = ScheduleConfig(
    window=K_SECONDS,  # Use optimal K for this currency
    alpha=0.1,         # 10% model, 90% TWAP blend (conservative — model signal is noisy)
    price_cap=3.0,     # Cap extreme exponent values
)

scheduler_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Use predictive_scheduler to build positions
    mid_preds = val_preds[i]
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,
        liq_pred=None,
        cfg=sched_cfg
    )
    
    # Simulate
    sched_vol, sched_avg_price = simulate_walk_the_book(
        scheduler_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if sched_vol > 0 and not np.isnan(sched_avg_price):
        impl_error = np.abs(sched_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / sched_vol)
        scheduler_bps.append(impl_error * vol_penalty)

scheduler_bps = np.array(scheduler_bps)

print(f"\n{'='*50}")
print(f"PREDICTIVE SCHEDULER IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Config: window={sched_cfg.window}, alpha={sched_cfg.alpha} (directional)")
print(f"Instruments evaluated: {len(scheduler_bps)}")
print(f"Mean:   {scheduler_bps.mean():.4f} bps")
print(f"Median: {np.median(scheduler_bps):.4f} bps")
print(f"Std:    {scheduler_bps.std():.4f} bps")
print(f"Min:    {scheduler_bps.min():.4f} bps")
print(f"Max:    {scheduler_bps.max():.4f} bps")
print(f"{'='*50}")


PREDICTIVE SCHEDULER IMPLEMENTATION ERROR
Config: window=14, alpha=0.1 (directional)
Instruments evaluated: 141
Mean:   1.3243 bps
Median: 1.1832 bps
Std:    1.0960 bps
Min:    0.0001 bps
Max:    9.0995 bps


## Positions

In [19]:
# Generate test positions for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Same config as evaluation
sched_cfg = ScheduleConfig(
    window=K_SECONDS,
    alpha=0.1,
    price_cap=3.0,
)

test_positions_scheduler = pd.DataFrame()

for i, anon_id in enumerate(test_ids):
    
    mid_preds = test_preds[i]
    
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,
        liq_pred=None,
        cfg=sched_cfg
    )

    df = pd.DataFrame({
        'anonymized_id': anon_id,
        'time_in_hour': times,
        'position': scheduler_positions
    })

    test_positions_scheduler = pd.concat([test_positions_scheduler, df], ignore_index=True)

test_positions_scheduler

,anonymized_id,time_in_hour,position
0,5.692284e+16,0 days 00:59:00,0.000000
1,5.692284e+16,0 days 00:59:01,0.000000
2,5.692284e+16,0 days 00:59:02,0.000000
3,5.692284e+16,0 days 00:59:03,0.000000
4,5.692284e+16,0 days 00:59:04,0.000000
...,...,...,...
18175,1.841373e+19,0 days 00:59:55,0.289109
18176,1.841373e+19,0 days 00:59:56,0.290852
18177,1.841373e+19,0 days 00:59:57,0.292682
18178,1.841373e+19,0 days 00:59:58,0.294603


# Save Positions

In [20]:
len(test_positions["anonymized_id"].unique())

303

In [21]:
base_path = Path(f"positions/{data_asset}")
base_path.mkdir(parents=True, exist_ok=True)

# 2. Map the filenames to the actual DataFrame objects
save_map = {
    "midprice.parquet": test_positions,
    "twap.parquet": test_positions_twap,
    "scheduler.parquet": test_positions_scheduler
}

# 3. Save each with a quick confirmation
for filename, df in save_map.items():
    if df is not None and not df.empty:
        target_path = base_path / filename
        df.to_parquet(target_path, index=False, engine='pyarrow')
        print(f"✅ Saved {len(df)} rows to: {target_path}")
    else:
        print(f"⚠️ Skipping {filename}: DataFrame is empty or None.")

✅ Saved 18180 rows to: positions/BTCUSDT/midprice.parquet
✅ Saved 18180 rows to: positions/BTCUSDT/twap.parquet
✅ Saved 18180 rows to: positions/BTCUSDT/scheduler.parquet


# Evaluation Metrics

We want a file of this format:

MSE, R2, BPS_MID, BPS_TWAP, BPS_SCHEDULER 

In [22]:
model.eval()

# Per-instrument normalization stats for VALIDATION set
mid_idx = {col: i for i, col in enumerate(FEATURE_COLS)}["mid_price"]
val_mid_mean = train_scalers["val_feat_means"][:, :, mid_idx]  # [1, num_val_ids]
val_mid_std  = train_scalers["val_feat_stds"][:, :, mid_idx]   # [1, num_val_ids]

# --- Step 1: Collect all targets and predictions ---
all_targets = []
all_preds = []

with torch.no_grad():
    for x_batch, y_batch, target in val_loader:
        x_batch = x_batch.to(config.device)
        pred = model(x_batch, y_teacher=None).cpu()  # [B, 60]
        target = target.cpu()                          # [B, 60]
        
        all_targets.append(target)
        all_preds.append(pred)

all_targets = torch.cat(all_targets)  # [num_val_ids, 60]
all_preds = torch.cat(all_preds)      # [num_val_ids, 60]

# --- Step 2: Denormalize to raw dollar space ---
mid_m = val_mid_mean.squeeze(0).unsqueeze(1).cpu()  # [num_val_ids, 1]
mid_s = val_mid_std.squeeze(0).unsqueeze(1).cpu()   # [num_val_ids, 1]

targets_denorm = all_targets * mid_s + mid_m
preds_denorm = all_preds * mid_s + mid_m

# --- Step 3: Per-instrument R2 and MSE (raw dollar space) ---
y_mean = targets_denorm.mean(dim=1, keepdim=True)  # mean PER instrument
ss_res = ((targets_denorm - preds_denorm) ** 2).sum(dim=1)  # PER instrument
ss_tot = ((targets_denorm - y_mean) ** 2).sum(dim=1)  # PER instrument
r2_per_inst = 1 - ss_res / ss_tot
final_r2 = r2_per_inst.mean().item()

final_mse = ((preds_denorm - targets_denorm) ** 2).mean().item()

print(f"Denormalized MSE: {final_mse:.6f}")
print(f"Denormalized R2:  {final_r2:.6f}")

metrics_df = pd.DataFrame({
    'data_asset': [data_asset],
    'mse': [final_mse],
    'r2': [final_r2],
    'bps_mid': [model_bps.mean().item()],
    'bps_twap': [baseline_bps.mean().item()],
    'bps_scheduler': [scheduler_bps.mean().item()]
})

target_path = base_path / "eval_df.parquet"
metrics_df.to_parquet(target_path, index=False, engine='pyarrow')
print(f"\nSaved eval metrics to: {target_path}")

Denormalized MSE: 2502.436523
Denormalized R2:  -14.232152

Saved eval metrics to: positions/BTCUSDT/eval_df.parquet


# Test Predictions

In [23]:
from utils.test import generate_test_predictions

# --- Execution ---
test_preds, test_id_map = generate_test_predictions(model, config, processor, num_ids=5)

Loading test data...
Sampling first 5 IDs for inference...
Preprocessing test data...
Running inference on 5 instruments...


* **The Map:** This `test_id_np` array pairs the model's row index with the actual anonymized ID.

In [24]:
# Convert test_id_map to numpy (handles both CPU and GPU tensors)
if hasattr(test_id_map, 'cpu'):
    test_id_np = test_id_map.cpu().numpy().astype(np.uint64)
else:
    test_id_np = np.array(test_id_map, dtype=np.uint64)

# Create a quick summary for inspection
print(f"{'Anonymized ID':<25} | {'First Pred (t+1)':<18} | {'Last Pred (t+60)':<18}")
print("-" * 70)

for i in range(len(test_id_np)):  # Look at first 5
    orig_id = test_id_np[i, 1]
    first_p = test_preds[i, 0]
    last_p  = test_preds[i, -1]
    print(f"{str(orig_id):<25} | {first_p:18.6f} | {last_p:18.6f}")

Anonymized ID             | First Pred (t+1)   | Last Pred (t+60)  
----------------------------------------------------------------------
1852841602367915877       |          -0.855928 |          -1.115844
4203491791199284318       |           0.953781 |           1.339514
8818603673003572008       |           0.746101 |           1.016212
12227300742381602074      |           0.334828 |           0.339883
13466858785193096784      |           0.409543 |           0.437988


In [25]:
# 1. Load your data into Polars
# Using test_id_np[:, 1] for our big IDs
id_df = pl.DataFrame({
    "anonymized_id": test_id_np[:, 1],
    "preds": test_preds.tolist()  # This creates a column of lists, each 60 items long
})

# 2. Explode and add the time duration
final_df = (
    id_df.explode("preds")  # This turns each list of 60 into 60 separate rows
    .with_columns(
        # This counts 0 to 59 for every ID
        pl.col("anonymized_id").cum_count().over("anonymized_id").sub(1).alias("step")
    )
    .with_columns(
        # Create the duration starting at 59 minutes
        pl.duration(minutes=59, seconds=pl.col("step")).alias("time_in_hour")
    )
    .select([
        "anonymized_id",
        "time_in_hour",
        pl.col("preds").alias("mid_price_pred")
    ])
)

display(final_df.head(65))

anonymized_id,time_in_hour,mid_price_pred
u64,duration[μs],f64
1852841602367915877,59m,-0.855928
1852841602367915877,59m 1s,-1.28406
1852841602367915877,59m 2s,-1.367718
1852841602367915877,59m 3s,-1.357563
1852841602367915877,59m 4s,-1.330785
…,…,…
4203491791199284318,59m,0.953781
4203491791199284318,59m 1s,1.420237
4203491791199284318,59m 2s,1.510006


# Final Comparison of Position Prediction

In [26]:
test_preds.shape

(5, 60)

In [27]:
bla = pd.read_parquet(X_test_path).sort_values(["anonymized_id", "time_in_hour"])
bla["anonymized_id"].unique().shape

(303,)

In [28]:
final_df.shape[0] / 60

5.0